##   Ручная загрузка меток

При получения конкретной ошибки классификации можно вручную залить нужную метку для конкреткного id платежа:
- никакой модели, никакого обучения и проверок,
- скачиваем нужный период по дате, находим ошибочный платеж, проверяем,
- генерируем запись с верной меткой и заливаем в БД. 

In [1]:
import pandas as pd
import numpy as np
import os
import requests

# ⏳ прогресс-бары
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

### Загрузим данные

In [2]:
# загружаем новые платежи за вчера

url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

start_date = "2025-09-15"
end_date = "2025-09-17"


params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming",
    "date_from": start_date,
    "date_to": end_date  
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"Данные загружены с сервера. Количество записей: {len(data_full)}")


Загружено страниц: 1 стр [00:02,  2.92s/ стр]

Данные загружены с сервера. Количество записей: 3166


### Метки категорий
"пожертвования от физических лиц (напрямую)": **1**  
"пожертвования через платформы": **2**  
"пожертвования от юридических лиц (напрямую)": **3**  
"прочие недоходные операции": **4**  
"продажа услуг": **5**  
"продажа товаров": **6**  
"финансовые доходы": **7**  
"членские и учредительские взносы": **8**  
"гранты субсидии конкурсы": **9**


In [3]:
data_full.head()

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id
0,1056179,1035,113805,2025-09-15,4000.0,None,33689,incoming,2192,0,0,-1,7184,1035,792,33689.0,792.0,32398.0,Оргвзнос при поступлении,2192.0,792.0,2190.0,Подростковая,7184.0,792.0,7181.0,...массовые,NaN,NaN,NaN,8.0
1,1056608,1034,52866,2025-09-15,23000.0,None,32395,incoming,2192,0,0,-1,7184,1034,792,32395.0,792.0,32393.0,Оплата за обучение,2192.0,792.0,2190.0,Подростковая,7184.0,792.0,7181.0,...массовые,NaN,NaN,NaN,5.0
2,1056669,714,102895,2025-09-15,88.0,Благотворительное пожертвование на уставную деятельность. НДС не облагается,13487,incoming,0,0,0,1729,6124,714,272,13487.0,272.0,13486.0,Пожертвование,NaN,NaN,NaN,None,6124.0,272.0,6122.0,Массовые разовые,1729.0,272.0,NaN,1.0
3,1056848,90,4908,2025-09-15,250.0,Аренда Селигерска Галя,6041,incoming,410,0,0,-1,1421,90,138,6041.0,138.0,6039.0,Услуги,410.0,138.0,0.0,Селигерская,1421.0,138.0,1373.0,Приносящая доход деятельность,NaN,NaN,NaN,5.0
4,1057151,2165,51,2025-09-15,100.0,ЗА 15/09/2025;Добровольное пожертвование;ЕЛИЗАРЬЕВА МАРИЯ ОЛЕГОВНА;ma.happy@yandex.ru;,37043,incoming,3529,0,0,5852,8396,2165,877,37043.0,877.0,37031.0,Сбербанк приложение,3529.0,877.0,3008.0,Общие поступления,8396.0,877.0,8393.0,...массовые,5852.0,877.0,NaN,1.0


In [ ]:
manual_uc_map = {
    1056179: 8,
    1056608: 5,
    1056669: 1,
}

In [ ]:
# генерируем запрос на запись в БД
payload = {
    "items": [
        {
            "payment_id": int(payment_id),
            "uc_id": int(uc_id)
        }
        for payment_id, uc_id in manual_uc_map.items()
    ]
}

In [ ]:
# отправляем запрос на запись в БД
url_up = "https://api.lemonpie.tech/api/payments/article-u"
headers = {
    "Authorization": "Bearer YOUR_API_TOKEN",
    "Content-Type": "application/json",
}

response = requests.post(url_up, json=payload, headers=headers)
print(f"Статус загрузки в БД: {response.status_code}, {response.text}")
print(f"✅ Завершение скрипта")

13:12:52 - INFO - Статус загрузки в БД: 200, {"status":"ok","inserted":0,"updated":493,"total":493}
13:12:52 - INFO - ✅ Завершение скрипта
